# 4주차 — Model I/O (1) 로컬 LLM과 상용 LLM을 하나의 인터페이스로 (Colab판)

「최신인공지능」 2026 · 4주차 실습 · 2026년 9월 18일 (금)

> **"코드 한 줄만 바꿉니다. 나머지는 그대로입니다."**
> 3주차에 만든 체인에서 **모델 자리만** 교체합니다.

| 실습 | 교시 | 내용 |
|------|------|------|
| 확인 B | 1교시 | VRAM 요구량 어림 계산 |
| 실습 0 ★ | 1교시 | Modelfile로 나만의 챗봇 — 코드 설정이 이긴다 |
| — | 2교시 | `ChatOllama` 파라미터 지정 / `.bind()` |
| 실습 1 ★★ | 2교시 | **동일 체인에서 모델 2종 교체·측정** |
| 실습 2 ★ | 2교시 | 6항목 비교표 작성 |
| — | 3교시 | `temperature` · `max_tokens` · `timeout` · `max_retries` |
| 실습 3 | 3교시 | 상용 실패 → 로컬 대체 (`with_fallbacks`) |
| 실습 4 ★ | 3교시 | 첫 토큰 지연 측정 → 비교표 완성 |

> ### ⚠️ Colab 과 실습실의 차이 — 1교시에 반드시 짚을 것
>
> 이 차시는 **"내 하드웨어 기준으로 판단한다"** 가 주제입니다.
> 그런데 **Colab T4 의 VRAM 은 15GB, 실습실 PC 는 8GB** 입니다.
>
> | | 실습실 PC | Colab T4 |
> |---|---|---|
> | VRAM | **8GB** | 15GB |
> | 12B 모델 | 경계~초과 | 여유 |
> | `ollama ps` 의 PROCESSOR | GPU | GPU (또는 GPU 미배정 시 100% CPU) |
>
> 아래 VRAM 계산 셀은 **두 기준을 나란히** 출력합니다.
> **비교표에 적을 값은 "내가 배포할 환경"(=8GB) 기준입니다.**

## 0. 환경 준비

In [ ]:
# ══════════════════════════════════════════════════════════════
#  Colab 환경 준비 — 매 세션 1회 실행 (재실행 안전)
# ══════════════════════════════════════════════════════════════
WEEK_MODELS   = ["chat", "small"]             # 실습 1 이 2종 비교라 소형도 받습니다
WEEK_PACKAGES = "langchain langchain-core langchain-ollama langchain-openai python-dotenv"
WEEK_SECRETS  = ["OPENAI_API_KEY"]            # 없으면 자동으로 로컬 2종 비교로 전환됩니다

# ──────────────────────────────────────────────────────────────
import os, shutil, subprocess, sys, time, urllib.request

IN_COLAB = "google.colab" in sys.modules
def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

GPU   = shutil.which("nvidia-smi") is not None and sh("nvidia-smi").returncode == 0
CHAT  = os.environ.setdefault("MODEL",       "gemma3:4b" if GPU else "gemma3:1b")
SMALL = os.environ.setdefault("SMALL_MODEL", "gemma3:1b")
EMBED = os.environ.setdefault("EMBED_MODEL", "nomic-embed-text")
TOOL  = os.environ.setdefault("TOOL_MODEL",  "qwen3:4b")
PICK  = {"chat": CHAT, "small": SMALL, "embed": EMBED, "tool": TOOL}

print(f"[1/5] 런타임   {'GPU 있음 ✅' if GPU else 'CPU 전용 ⚠️'}   →  대화 모델 {CHAT}")
if not GPU:
    print("       [런타임] > [런타임 유형 변경] > T4 GPU 로 바꾸면 4b 모델을 쓸 수 있습니다.")

print("[2/5] 패키지 설치 중…")
r = sh(f"{sys.executable} -m pip install -q {WEEK_PACKAGES}")
print("       ✅ 완료" if r.returncode == 0 else "       ❌ 실패\n" + r.stderr[-600:])

if shutil.which("ollama") is None:
    print("[3/5] Ollama 설치 중… (약 30초)")
    sh("curl -fsSL https://ollama.com/install.sh | sh")
print("[3/5] Ollama  " + ("✅ 준비됨" if shutil.which("ollama") else "❌ 설치 실패"))

def alive():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if not alive():
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        if alive():
            break
        time.sleep(1)
print("[4/5] 서버    " + ("✅ 응답함" if alive() else "❌ 미응답 — 이 셀을 다시 실행하세요"))

have = {ln.split()[0] for ln in sh("ollama list").stdout.splitlines()[1:] if ln.strip()}
for key in WEEK_MODELS:
    name = PICK[key]
    if name in have:
        print(f"[5/5] {name:<20s} ✅ 이미 있음")
        continue
    print(f"[5/5] {name:<20s} ⏳ 내려받는 중… (진행 표시 없이 수 분 걸립니다)")
    t0 = time.time()
    r = sh(f"ollama pull {name}")
    print(f"       {'✅ 완료' if r.returncode == 0 else '❌ 실패'}  ({time.time() - t0:.0f}초)")
    if r.returncode != 0:
        print(r.stderr[-400:])

for k in WEEK_SECRETS:
    if not os.getenv(k) and IN_COLAB:
        try:
            from google.colab import userdata
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
    print(f"[키]  {k:<20s} " + ("✅ 설정됨" if os.getenv(k) else "⬜ 없음 (없어도 진행됩니다)"))

print("\n" + "=" * 62)
print(f"준비 완료 — MODEL='{CHAT}'  SMALL_MODEL='{SMALL}'")
print("=" * 62)

## 1교시 — 모델을 조회하고, 내 하드웨어에 올라가는지 판단한다

실습실에서는 터미널에서 직접 칩니다. Colab에서는 `!` 로 같은 명령을 실행합니다.

In [ ]:
# 받아 놓은 모델 목록
!ollama list

In [ ]:
# ★ 파라미터 수 · 컨텍스트 길이 · 양자화 조회 — 아래 계산의 입력값이 여기 있습니다
import os
!ollama show {os.environ["MODEL"]}

### 확인 B — VRAM 요구량 어림 계산

**손으로 먼저 계산하고, 그 다음 이 셀로 답을 맞춰 봅니다.**

```
필요 VRAM = ① 가중치 + ② KV 캐시 + ③ 실행 오버헤드

① 가중치(GB) = 파라미터 수(B) × 실효 비트수 ÷ 8
                Q4_K_M 의 실효 비트는 4가 아니라 약 4.5    ★
② KV 캐시     = 파라미터 1B · 컨텍스트 1K 당 대략 12~13MB
③ 오버헤드    = CUDA 컨텍스트·런타임으로 0.5 ~ 1GB
```

⚠️ 전부 어림값입니다. 정확한 값이 목적이 아니라
**"올라가는가 / 아슬아슬한가 / 안 되는가"를 3초 안에 가르는 것**이 목적입니다.

In [ ]:
import subprocess

# ── 실습실 PC 기준 (비교표에 적을 값은 이쪽입니다) ★ ──
LAB_VRAM_GB = 8.0

# ── 지금 이 Colab 런타임의 실제 VRAM ──
def detect_vram() -> float | None:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10,
        )
        return int(out.stdout.strip().splitlines()[0]) / 1024
    except Exception:
        return None

COLAB_VRAM_GB = detect_vram()

EFFECTIVE_BITS   = 4.5      # Q4_K_M 의 실효 비트수 — 중요한 부분에 더 높은 비트를 섞으므로 4가 아니다 ★
KV_GB_PER_B_PER_K = 0.0125  # 파라미터 1B · 컨텍스트 1K 당 KV 캐시 (GB) 어림값
OVERHEAD_GB      = 0.7      # CUDA 컨텍스트 등 실행 오버헤드


def estimate(params_b: float, ctx_tokens: int, bits: float = EFFECTIVE_BITS) -> dict:
    """파라미터 수(B)와 컨텍스트 길이(토큰)로 필요 VRAM을 어림한다."""
    weights = params_b * bits / 8                                    # ① 가중치
    kv      = params_b * (ctx_tokens / 1024) * KV_GB_PER_B_PER_K     # ② KV 캐시
    return {"weights": weights, "kv": kv, "overhead": OVERHEAD_GB,
            "total": weights + kv + OVERHEAD_GB}


def verdict(total_gb: float, vram: float) -> str:
    """주어진 VRAM 에서 어떻게 되는가 — 세 갈래로만 가른다."""
    if total_gb <= vram * 0.8:
        return "[가능] 여유 있음"
    if total_gb <= vram * 1.1:
        # 어림값이므로 VRAM 근처(±10%)는 '경계'로 본다.
        # 에러가 아니라 '느려지는' 구간이라 이게 더 위험하다 ★
        return "[경계] 넘치기 쉬움 - 일부 레이어가 CPU로 밀릴 수 있음"
    return "[초과] CPU 분산으로 수 배 느려짐"


def table(ctx: int) -> None:
    print(f"── 컨텍스트 {ctx // 1024}K 기준 " + "─" * 40)
    for label, params_b in [("4B", 4.0), ("8B", 8.0), ("12B", 12.0)]:
        r = estimate(params_b, ctx)
        line = (f"  {label:<5} 가중치 {r['weights']:>5.1f}GB  KV {r['kv']:>4.1f}GB  "
                f"오버헤드 {r['overhead']:>4.1f}GB  = 합계 {r['total']:>5.1f}GB")
        print(line)
        print(f"        └ 실습실 8GB   {verdict(r['total'], LAB_VRAM_GB)}   ★ 비교표에는 이 판정을")
        if COLAB_VRAM_GB:
            print(f"        └ Colab {COLAB_VRAM_GB:.0f}GB  {verdict(r['total'], COLAB_VRAM_GB)}")


print(f"양자화 Q4_K_M (실효 {EFFECTIVE_BITS}비트)")
print(f"실습실 VRAM {LAB_VRAM_GB:.0f}GB / 지금 런타임 "
      + (f"{COLAB_VRAM_GB:.0f}GB" if COLAB_VRAM_GB else "GPU 없음"))
print()
table(4096)
print()
table(8192)      # ← 같은 모델, 컨텍스트만 8K 로 늘리면 (num_ctx 의 대가) ★

print()
print("[정리] 실습실 8GB 기준에서 12B는 '되긴 하는데 느린' 구간입니다.")
print("   인터넷 가이드가 권하는 모델이라도, 판단은 내 하드웨어 기준으로.")
print("   ⚠️ Colab 15GB 에서 여유롭다고 실습실에서도 여유로운 것이 아닙니다. ★")

In [ ]:
# ── 내 화면의 값으로 계산하기 ──
# 파라미터 수는 `ollama show` 의 parameters, 컨텍스트는 `ollama ps` 의 CONTEXT(실제 창)를 넣으십시오.
# ⚠️ `ollama show` 의 context length(131072)는 최대치입니다 — 넣으면 틀린 결론이 나옵니다.
PARAMS_B = 4.3
CTX      = 4096

r = estimate(PARAMS_B, CTX)
print(f"파라미터 {PARAMS_B}B / 컨텍스트 {CTX}")
print(f"  가중치 {r['weights']:.1f}GB + KV {r['kv']:.1f}GB + 오버헤드 {r['overhead']:.1f}GB "
      f"= {r['total']:.1f}GB")
print(f"  실습실 8GB : {verdict(r['total'], LAB_VRAM_GB)}")

In [ ]:
# 모델을 메모리에 올리고 실제 점유를 확인한다
import os
!ollama run {os.environ["MODEL"]} "안녕" > /dev/null 2>&1
print("── ollama ps — SIZE 와 PROCESSOR 열을 보세요 ★ ──")
!ollama ps

> **`ollama ps` 에서 볼 것**
>
> | 열 | 의미 |
> |---|---|
> | `SIZE` | 실제 메모리 점유 — 위 계산값과 대조하십시오 ★ |
> | `PROCESSOR` | `100% GPU` 면 정상. `xx% CPU` 가 섞이면 VRAM 초과로 밀린 것 |
>
> GPU 런타임이 아니면 `100% CPU` 로 나옵니다. 그 상태로도 실습은 되지만 매우 느립니다.

## 실습 0 (1교시 §3) — Modelfile로 나만의 챗봇

실습실에서는 `code/Modelfile` 을 VS Code 로 열고 터미널에서 `ollama create` 합니다.
Colab에서는 셀에서 파일을 만들어 **같은 명령**을 실행합니다.

| 지시어 | 하는 일 | 오늘 |
|---|---|---|
| `FROM` | 어떤 모델에서 출발하나 — 필수 · 맨 위 | ★ |
| `SYSTEM` | 모든 대화 앞에 붙는 역할 지시 | ★ |
| `PARAMETER` | 기본 설정값 (`temperature` · `num_ctx` …) | ★ |
| `MESSAGE` | 예시 대화 (user / assistant) | 씀 |
| `TEMPLATE` · `ADAPTER` | 대화 서식 · LoRA | ✕ |

> ⚠️ **Modelfile 은 모델을 다시 학습시키지 않습니다.** 가중치는 그대로이고,
> **앞에 붙일 글과 기본값**만 저장합니다. 그래서 `create` 는 1초, 디스크는 약 1KB 입니다.

In [ ]:
# 원본 모델의 레시피 카드 — FROM · TEMPLATE · PARAMETER · LICENSE
import os
!ollama show {os.environ["MODEL"]} --modelfile | grep -E '^(FROM|PARAMETER|TEMPLATE|LICENSE|SYSTEM)'

In [ ]:
# ── 실습 0-① Modelfile 만들기 ──────────────────────────
#    FROM 은 이 런타임의 대화 모델을 씁니다 (GPU 면 gemma3:4b, CPU 면 gemma3:1b)
import os, pathlib

MODELFILE = f'''FROM {os.environ["MODEL"]}

SYSTEM """
당신은 '파이봇'입니다. 파이썬을 처음 배우는 대학생을 돕는 튜터입니다.
- 한국어로, 다섯 문장 이내로 답합니다.
- 코드는 10줄 이내 예제 하나만 보여 줍니다.
- 과제 정답을 통째로 주지 말고, 먼저 힌트를 줍니다.
"""

PARAMETER temperature 0.3
PARAMETER num_ctx 4096
PARAMETER num_predict 400

MESSAGE user 변수가 뭐예요?
MESSAGE assistant 변수는 값에 붙이는 이름표예요. age = 20 이라고 쓰면 20에 age라는 이름표가 붙고, 이후 age를 부르면 20이 나옵니다. 직접 name 변수를 하나 만들어 볼까요?
'''
pathlib.Path("Modelfile").write_text(MODELFILE, encoding="utf-8")   # ★ UTF-8 로 저장
print(MODELFILE)

In [ ]:
# 만들고 → 확인하고 → 물어본다
!ollama create py-tutor -f Modelfile 2>/dev/null && echo "✅ create 완료"
!ollama show py-tutor
!ollama run py-tutor "리스트가 뭐야?" 2>/dev/null

### 실습 0-② 나만의 챗봇으로 바꾸기

위 셀의 `SYSTEM` · `PARAMETER` · `MESSAGE` 를 고치고, 이름을 바꿔 다시 만드세요.

| 항목 | 조건 |
|---|---|
| 이름 | 영어 소문자 · 숫자 · 하이픈 (`interview-bot`) — 한글 이름은 `invalid model name` |
| `SYSTEM` | 역할 + **규칙 세 개 이상** (말투 · 길이 · 금지) |
| `PARAMETER` | `temperature` 를 역할에 맞게 |
| `MESSAGE` | 예시 대화 한 쌍 이상 |

```
!ollama create my-bot -f Modelfile
!ollama run my-bot "질문"
```

In [ ]:
# ── 실습 0-③ 코드에서 부르면 — 누가 이기나 ★★ ─────────────
#    답의 '내용'이 아니라 '토큰 수'를 봅니다 (code/my_bot.py 와 같은 실험)
import os
from langchain_ollama import ChatOllama

BASE, BOT = os.environ["MODEL"], "py-tutor"
Q = [("human", "리스트가 뭐야?")]
S = [("system", "당신은 친절한 비서입니다."), ("human", "리스트가 뭐야?")]

def tokens(model, messages, **params):
    r = ChatOllama(model=model, **params).invoke(messages)
    u = r.usage_metadata or {}
    return u.get("input_tokens"), u.get("output_tokens"), r.content

b1, _, _ = tokens(BASE, Q, num_predict=1)
p1, _, _ = tokens(BOT, Q, num_predict=1)
b2, _, _ = tokens(BASE, S, num_predict=1)
p2, _, _ = tokens(BOT, S, num_predict=1)
_, out3, ans3 = tokens(BOT, Q, num_predict=20)

print(f"① 그냥 부르기        입력 토큰  {BASE} {b1:>4}  →  {BOT} {p1:>4}   (+{p1 - b1} = SYSTEM + MESSAGE)")
print(f"② 코드에서 system    입력 토큰  {BASE} {b2:>4}  →  {BOT} {p2:>4}   (+{p2 - b2} = MESSAGE 만 → SYSTEM 교체됨)")
print(f"③ 코드 num_predict=20 출력 토큰 {out3}  → Modelfile 의 400 을 덮어씀")
print("   ", ans3)

> **Modelfile 은 '기본값'입니다. 코드에서 주면 코드가 이깁니다.**
>
> | 코드에서 준 것 | Modelfile 쪽은 |
> |---|---|
> | `system` 메시지 | `SYSTEM` 이 빠짐 — **교체** (MESSAGE 예시는 남음) |
> | `num_predict` · `temperature` 등 | `PARAMETER` 를 **덮어씀** |
>
> 2교시에 `ChatOpenAI` 로 바꾸면 파이봇의 설정은 **따라가지 않습니다** → 모델을 갈아끼울 코드라면 **설정은 코드에**.
>
> 자료 제작 PC 실측(gemma3:4b · Ollama 0.34.0): ① 15 → 172 · ② 31 → 100 · ③ 20.
> Colab 의 모델이 1b 라면 숫자는 달라도 **차이의 방향**은 같습니다.

## 2교시 1절 — `ChatOllama` 에 파라미터 붙이기

지난주에는 `ChatOllama(model=...)` 만 썼습니다. 오늘은 여기에 설정을 붙입니다.

- **A) 생성자에 지정** — 그 객체 전체에 적용
- **B) `.bind()` 로 덧붙이기** — 모델 객체 하나로 설정만 다른 체인을 파생시킬 때 ★
  (같은 방식이 9주차 도구 호출 `bind_tools` 에서 다시 나옵니다)

> ### ⚠️ 함정 — 같은 `.bind()` 인데 공급자마다 받는 모양이 반대입니다 ★★
>
> **공급자** = 모델을 실제로 돌려 주는 쪽. 오늘은 둘 — **Ollama**(이 노트북에서 띄운 서버) · **OpenAI**(상용 API).
> `.bind()` 는 값을 **각 회사 라이브러리에 그대로 넘기므로**, 받는 모양이 다르면 결과가 반대가 됩니다.
>
> | | Ollama — `ChatOllama` | OpenAI — `ChatOpenAI` |
> |---|---|---|
> | `.bind()` 값을 넘겨받는 함수 | `ollama` 의 `Client.chat()` | `openai` 의 `chat.completions.create()` |
> | `temperature` 를 받는 자리 | `options={...}` 묶음 **안쪽** | **맨 바깥** 인자 |
> | `bind(temperature=0.9)` | ❌ `TypeError` | ✅ 됨 |
> | `bind(options={"temperature": 0.9})` | ✅ 됨 | ❌ `TypeError` |
>
> ```python
> base.bind(temperature=0.9)                 # Ollama → TypeError 로 죽습니다
> base.bind(options={"temperature": 0.9})    # Ollama 는 이렇게 써야 합니다
> ```

In [ ]:
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

MODEL = os.environ["MODEL"]

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 프로그래밍 강사입니다. {level} 눈높이로 설명하세요."),
        ("human",  "{topic}의 장점 3가지를 각각 한 문장으로 알려줘."),
    ]
)
parser = StrOutputParser()
INPUTS = {"level": "초보자", "topic": "파이썬"}

# ── 방법 A: 생성자에 지정 ─────────────────────────────
llm = ChatOllama(
    model=MODEL,
    temperature=0.2,   # 1교시에서 본 그 파라미터
    num_predict=300,   # 최대 생성 토큰 수
    num_ctx=4096,      # 컨텍스트 창 크기  ← VRAM에 영향 ★ (1교시 계산)
)
print("=" * 60)
print("[방법 A] 생성자에 지정 - temperature=0.2")
print("=" * 60)
print((prompt | llm | parser).invoke(INPUTS))

In [ ]:
# ── 방법 B: .bind() 로 나중에 덧붙이기 ────────────────
#    모델 객체는 하나. 설정만 다른 체인을 두 개 파생시킨다.
#
#    ⚠️ ChatOllama 에서는 options={...} 로 감싸야 합니다.  ★★
#    ⚠️ options 는 '덧붙이기'가 아니라 '통째 교체'입니다.
#       생성자에서 준 num_predict·num_ctx 를 유지하려면 여기 같이 적어야 합니다.

base     = ChatOllama(model=MODEL)
creative = base.bind(options={"temperature": 0.9, "num_predict": 300})
strict   = base.bind(options={"temperature": 0.0, "num_predict": 300})

for label, bound in [("창의적 temperature=0.9", creative), ("엄격 temperature=0.0", strict)]:
    print("=" * 60)
    print(f"[방법 B] .bind() - {label}")
    print("=" * 60)
    print((prompt | bound | parser).invoke(INPUTS))
    print()

print("[주의] num_predict 는 Ollama 고유 이름, OpenAI 는 max_tokens 입니다.")
print("       .bind() 도 마찬가지 - Ollama 는 options={...} 로 감싸야 하고,")
print("       OpenAI 는 bind(temperature=0.9) 가 그대로 됩니다.")
print("       LangChain이 '모든 것'을 통일해 주지는 않습니다. (3교시 1절에서 정리)")

## 실습 1 ★★ (2교시) — 같은 체인에 모델만 갈아끼우고 측정한다

```
chain = prompt | llm | parser
                  ▲
                  │
           여기만 바꾼다
    ┌─────────────┴─────────────┐
ChatOllama(로컬)          ChatOpenAI(상용)

prompt 그대로 · parser 그대로 · invoke 그대로     ← 오늘의 핵심 ★
```

**관찰 포인트**

1. 바뀐 코드는 아래 `build_models()` 의 **한 줄** 뿐이다 ★
2. 두 모델이 교체 가능한 이유 = 같은 Runnable 규약(`invoke`)을 따르기 때문
3. 총 소요 시간은 재지만 **'첫 토큰까지'는 `invoke` 로 잴 수 없다** → 3교시 실습 4에서 채운다 ★

> ⚠️ `ChatAnthropic` 은 쓰지 않습니다. Anthropic API 키가 필요하며,
> Claude Code 구독 계정으로는 호출할 수 없습니다.
>
> 🔶 `OPENAI_API_KEY` 가 없으면 **로컬 2종 비교로 자동 전환**됩니다. 코드 구조는 그대로입니다.

In [ ]:
import os, time
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI

LOCAL_MODEL = os.environ["MODEL"]
LOCAL_SMALL = os.environ["SMALL_MODEL"]      # 상용 키를 못 쓸 때의 대체 비교용

# 🔶 상용 모델명은 자주 바뀝니다. 수업 전날 공식 문서에서 확인해 확정할 것 ★
OPENAI_MODEL = "gpt-4o-mini"

# ── 프롬프트와 파서는 한 번만 만든다. 끝까지 바뀌지 않는다 ──
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 프로그래밍 강사입니다. {level} 눈높이로 설명하세요."),
        ("human",  "{topic}의 장점 3가지를 각각 한 문장으로 알려줘."),
    ]
)
parser = StrOutputParser()
INPUTS = {"level": "초보자", "topic": "파이썬"}


def build_models() -> dict:
    """바뀌는 것은 이 목록뿐입니다."""
    if os.getenv("OPENAI_API_KEY"):
        return {
            f"로컬 {LOCAL_MODEL}":     ChatOllama(model=LOCAL_MODEL, temperature=0.2),
            f"OpenAI {OPENAI_MODEL}": ChatOpenAI(model=OPENAI_MODEL, temperature=0.2),
        }

    # ── 대체안: 상용 키가 없으면 로컬 2종으로 비교한다 ──
    #    코드 구조는 완전히 같습니다. 비교표의 '비용' 칸을
    #    'ollama ps 의 SIZE(메모리 점유)' 로 바꿔서 채우면 됩니다. ★
    print("[주의] OPENAI_API_KEY 가 없습니다 → 로컬 2종 비교로 진행합니다.")
    print("   (Colab 좌측 🔑 보안 비밀에 키를 넣으면 자동으로 로컬 ↔ OpenAI 비교가 됩니다)")
    print()
    return {
        f"소형 {LOCAL_SMALL}": ChatOllama(model=LOCAL_SMALL, temperature=0.2),
        f"중형 {LOCAL_MODEL}": ChatOllama(model=LOCAL_MODEL, temperature=0.2),
    }


def run_one(name: str, llm) -> dict:
    """한 모델을 돌리고 측정값을 돌려준다."""
    # 💡 파서를 빼면 AIMessage 객체가 그대로 온다 → 토큰 사용량을 볼 수 있다 ★
    #    StrOutputParser 는 문자열만 남기고 메타데이터를 버립니다.
    chain_meta = prompt | llm

    t0 = time.perf_counter()
    msg = chain_meta.invoke(INPUTS)
    elapsed = time.perf_counter() - t0

    text  = parser.invoke(msg)
    usage = msg.usage_metadata or {}     # 🔶 공급자·버전에 따라 비어 있을 수 있음

    print("=" * 60)
    print(f"[{name}]  총 소요 {elapsed:.2f}초")
    print("=" * 60)
    print(text)
    print()
    print("  usage_metadata:", usage or "(측정 불가)")

    return {"name": name, "elapsed": elapsed,
            "input_tokens": usage.get("input_tokens"),
            "output_tokens": usage.get("output_tokens")}


rows = [run_one(name, llm) for name, llm in build_models().items()]

In [ ]:
print("=" * 60)
print("측정 요약 - 실습 2 비교표에 옮겨 적으세요")
print("=" * 60)
print(f"  {'모델':<24} {'총 소요':>8} {'입력':>8} {'출력':>8} {'첫 토큰':>10}")
for r in rows:
    i = r["input_tokens"]  if r["input_tokens"]  is not None else "측정불가"
    o = r["output_tokens"] if r["output_tokens"] is not None else "측정불가"
    print(f"  {r['name']:<24} {r['elapsed']:>7.2f}초 {i:>8} {o:>8} {'(3교시)':>10}")

print()
print("[대기] '첫 토큰까지' 칸은 invoke 로는 잴 수 없습니다.")
print("       3교시 실습 4에서 채웁니다. ★")
print()
print("[비용] (입력 토큰 × 입력 단가) + (출력 토큰 × 출력 단가)")
print("       단가는 외우지 마세요. 외울 것은 계산 구조 - 출력이 대체로 더 비쌉니다.")

> ### ⚠️ '상용이 항상 좋다'로 끝내지 마십시오
>
> 이 프롬프트처럼 단순한 작업은 로컬 4B로 충분한 경우가 많습니다.
> **작업 난이도에 따라 답이 달라진다**는 것이 요점입니다.
>
> 시간이 남으면 위 셀의 `INPUTS` 를 다단계 추론이 필요한 어려운 질문으로 바꿔
> 한 번 더 돌려 차이를 보십시오.

## 실습 2 ★ (2교시) — 비교표 작성

아래 표를 채워 저장소에 커밋하십시오. (`week04/비교표.md`)

| 항목 | 로컬 (`gemma3:4b`) | 상용 (`gpt-4o-mini`) |
|---|---|---|
| 총 소요 시간 | | |
| 입력 토큰 | | |
| 출력 토큰 | | |
| **첫 토큰까지** (3교시에 채움) | | |
| 비용 / 메모리 점유 | | |
| 답변 품질 (주관 평가) | | |

> 📌 비교표는 정규 과제가 아니지만 **버리지 마십시오.**
> **10주차 미니 프로젝트 명세**에서 *"왜 이 모델을 골랐는가"* 의 근거로 그대로 씁니다.
>
> ⚠️ '비용' 칸은 상용 키가 없으면 **'메모리 점유'(`ollama ps` 의 SIZE)** 로 치환합니다.

## 3교시 1절 — 공통 파라미터 4종

```
temperature · max_tokens      ← 모델에게 주는 지시
timeout · max_retries         ← 호출을 어떻게 할 것인가 (네트워크 계층)
```

뒤의 둘은 **로컬에서는 의미가 약합니다** — 인터넷을 안 타므로. 상용 API에서 중요합니다.

⚠️ `ChatOllama` 는 `max_tokens` · `timeout` · `max_retries` 를 **에러 없이 무시**합니다 — 로컬의 길이 제한은 `num_predict` 입니다.

In [ ]:
import os
from langchain_ollama import ChatOllama

MODEL    = os.environ["MODEL"]
QUESTION = "파이썬의 장점 3가지를 각각 한 문장으로 알려줘."

# ── ① temperature — 무작위성 ──
print("=" * 60)
print("① temperature - 무작위성")
print("=" * 60)

for temp in (0.0, 0.9):
    # num_predict 로 짧게 끊습니다 — 앞부분만 봐도 차이가 드러나고, 실습이 빨라집니다
    llm = ChatOllama(model=MODEL, temperature=temp, num_predict=60)
    print(f"\n── temperature={temp} : 같은 질문을 두 번 ──")
    for i in (1, 2):
        answer = llm.invoke(QUESTION).content.replace("\n", " ")
        print(f"  [{i}회] {answer[:70]}...")

print()
print("  0 ~ 0.3   : 분류·추출·요약·JSON 출력 - 일관성이 중요할 때 (5주차 구조화 출력)")
print("  0.7 ~ 1.0 : 아이디어 생성·창작")

In [ ]:
# ── ② 최대 생성 길이 ──
print("=" * 60)
print("② 최대 생성 길이 - Ollama: num_predict / OpenAI: max_tokens  [이름이 다름]")
print("=" * 60)

llm = ChatOllama(model=MODEL, temperature=0.2, num_predict=40)
print(llm.invoke(QUESTION).content)

print()
print("  [주의] '40토큰으로 요약해줘'가 아닙니다. 40토큰에서 잘립니다.")
print("     문장 중간에서 끊기죠? 길이를 조절하려면 프롬프트로도 함께 요청해야 합니다.")
print("     비용 상한을 거는 안전장치로 이해하는 것이 정확합니다.")

### ③ `timeout` / ④ `max_retries` — 호출을 어떻게 할 것인가

```python
ChatOpenAI(model="...", timeout=30)      # 30초 안에 응답 없으면 포기
ChatOpenAI(model="...", max_retries=2)   # 일시적 오류면 2번까지 다시 건다
```

> ⚠️ 둘 다 **상용 API(`ChatOpenAI`)용**입니다. `ChatOllama` 에 넣으면 **에러 없이 무시**됩니다.

```
    호출
      ├─ 성공 ─────────────────────▶ 끝
      └─ 실패 ─▶ max_retries 만큼 재시도
                    ├─ 성공 ────────▶ 끝
                    └─ 계속 실패 ─▶ 폴백(다른 모델)   ★ 실습 3
```

> **[주의]** 재시도가 듣는 것은 **'일시적' 오류뿐**입니다.
> 잘못된 키·없는 모델명은 몇 번을 걸어도 실패합니다. → 그때 필요한 것이 폴백.

## 실습 3 (3교시) — 상용 실패 → 로컬 대체 `with_fallbacks`

```
     상용 API 호출
          │
    실패(장애·한도 초과·키 문제)
          │
          ▼
  로컬 모델로 자동 전환      ← 서비스는 계속된다
```

**관찰 포인트**

1. 주 모델이 실패했는데 **에러 없이 결과가 나온다** ★
2. ⚠️ 그래서 장애가 **'조용히' 묻힙니다.**
   어떤 모델이 실제로 응답했는지 기록해야 합니다 → **6주차 LangSmith**

In [ ]:
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI

LOCAL_MODEL = os.environ["MODEL"]

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 프로그래밍 강사입니다. {level} 눈높이로 설명하세요."),
        ("human",  "{topic}의 장점 3가지를 각각 한 문장으로 알려줘."),
    ]
)
parser = StrOutputParser()
INPUTS = {"level": "초보자", "topic": "파이썬"}


def build_primary():
    """주 모델 — 일부러 실패하게 만든다 ★

    실패를 만드는 방법 세 가지
      ① 없는 모델명   model="gpt-존재하지-않는-모델"  ← 권장
      ② 잘못된 키     api_key="sk-틀린키"             ← 키를 망가뜨릴 위험
      ③ 극단적 타임아웃 timeout=0.001                 ← 재현이 들쭉날쭉
    """
    if os.getenv("OPENAI_API_KEY"):
        return "OpenAI(없는 모델명)", ChatOpenAI(
            model="gpt-존재하지-않는-모델",   # 실패 유발 ①
            timeout=10,
            max_retries=0,                    # 재시도 없이 바로 실패시켜 시간을 아낀다
        )

    # 키가 없어도 실습은 그대로 됩니다. 주 모델을 '없는 로컬 모델'로 바꾸면
    # 로컬 → 로컬 폴백이 되고, 관찰할 것은 완전히 같습니다.
    print("[주의] OPENAI_API_KEY 가 없습니다 → 주 모델을 '없는 로컬 모델'로 대체합니다.")
    print()
    return "로컬(없는 모델명)", ChatOllama(model="없는-모델-이름")


primary_name, primary = build_primary()

# ── 대체 모델: 로컬 ──
backup = ChatOllama(model=LOCAL_MODEL, temperature=0.2)

# ── 폴백 연결 ★ — 이 한 줄이 전부입니다 ──
llm = primary.with_fallbacks([backup])

print("=" * 60)
print(f"주 모델({primary_name})을 부릅니다 …")
print("=" * 60)

# 체인의 형태는 지금까지와 똑같습니다
#     chain = prompt | llm | parser
# 다만 여기서는 '누가 대답했는지'까지 보려고 파서를 빼고 호출합니다.
msg = (prompt | llm).invoke(INPUTS)
print(parser.invoke(msg))

# ── ⚠️ 폴백의 대가 — 누가 대답했는지 확인해 봅시다 ──────
meta   = msg.response_metadata or {}
actual = meta.get("model_name") or meta.get("model") or "(알 수 없음)"

print()
print("=" * 60)
print(f"실제로 응답한 모델: {actual}")
print("=" * 60)

| 폴백이 해결하는 것 | 해결하지 못하는 것 |
|---|---|
| 서비스가 멈추지 않는다 | 응답 품질이 떨어진다 (사용자는 모른 채 받는다) |
| 장애 대응 코드가 짧다 | 실패가 조용히 묻힌다 — 로그가 없으면 눈치 못 챔 |
| (없음) | 로컬 모델이 VRAM 에 안 올라가면 **폴백도 실패** ★ |

> **[주의]** 폴백이 걸려 있으면 장애가 안 보입니다.
> 그래서 '어떤 모델이 실제로 응답했는지'를 위처럼 기록해야 합니다.
> 이 기록을 자동으로 남겨 주는 도구가 **6주차 LangSmith** 입니다.

## 실습 4 ★ (3교시) — 첫 토큰까지의 시간(TTFT) 측정

```
[invoke]  질문 ──────────── (10초 침묵) ────────────▶ 답 전체가 한 번에
[stream]  질문 ─ 0.8초 ─▶ 답 ─ 이 ─ 조 ─ 금 ─ 씩 ─ 나 ─ 온 ─ 다 ▶ (총 10초)
```

**총 소요 시간은 같습니다. 달라지는 것은 '첫 글자가 언제 나오는가' 입니다.**

**관찰 포인트**

1. 바뀐 것은 `invoke` → `stream` 과 반복문뿐. 체인은 그대로다 ★
2. 총 시간 ≈ 변화 없음 → **스트리밍은 빨라지게 하지 않는다**
3. 첫 토큰까지 ≪ 총 시간 → 개선되는 것은 **'체감 속도'**
4. 여기서 잰 값으로 **2교시 비교표의 '첫 토큰까지' 칸을 채웁니다** ★★

> 🔶 로컬 모델이 메모리에서 내려가 있으면 첫 토큰까지가 크게 늘어납니다
> (모델 로딩 시간이 포함되므로). 바로 위 셀들을 먼저 실행해 모델을 올려 두십시오.

In [ ]:
import os, time
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI

LOCAL_MODEL  = os.environ["MODEL"]
OPENAI_MODEL = "gpt-4o-mini"       # 🔶 수업 전날 공식 문서에서 확인해 확정할 것

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 프로그래밍 강사입니다. {level} 눈높이로 설명하세요."),
        ("human",  "{topic}의 장점 3가지를 각각 한 문장으로 알려줘."),
    ]
)
INPUTS = {"level": "초보자", "topic": "파이썬"}


def measure_invoke(chain) -> float:
    """① invoke — 다 나올 때까지 기다린다."""
    t0 = time.perf_counter()
    chain.invoke(INPUTS)
    total = time.perf_counter() - t0
    print(f"[invoke] 총 {total:.2f}초 (그동안 화면은 조용했습니다)")
    return total


def measure_stream(chain) -> tuple:
    """② stream — 오는 대로 출력한다. 첫 조각이 온 시각을 기록한다."""
    t0, first = time.perf_counter(), None

    for chunk in chain.stream(INPUTS):        # ★ invoke → stream, 이것뿐입니다
        if first is None and chunk:
            first = time.perf_counter() - t0  # 첫 글자가 도착한 시각
        print(chunk, end="", flush=True)

    total = time.perf_counter() - t0
    ttft  = f"{first:.2f}초" if first is not None else "측정 불가"
    print(f"\n[stream] 첫 토큰까지 {ttft} / 총 {total:.2f}초")
    return first, total


def run(name: str, llm) -> dict:
    chain = prompt | llm | StrOutputParser()
    print("=" * 60)
    print(f"[{name}]")
    print("=" * 60)
    invoke_total = measure_invoke(chain)
    print()
    ttft, stream_total = measure_stream(chain)
    print()
    return {"name": name, "invoke_total": invoke_total,
            "ttft": ttft, "stream_total": stream_total}


models = {f"로컬 {LOCAL_MODEL}": ChatOllama(model=LOCAL_MODEL, temperature=0.2)}
if os.getenv("OPENAI_API_KEY"):
    models[f"OpenAI {OPENAI_MODEL}"] = ChatOpenAI(model=OPENAI_MODEL, temperature=0.2)
else:
    print("[주의] OPENAI_API_KEY 가 없어 로컬 모델만 측정합니다.\n")

srows = [run(name, llm) for name, llm in models.items()]

In [ ]:
print("=" * 60)
print("비교표의 '첫 토큰까지' 칸을 지금 채우세요  ★")
print("=" * 60)
print(f"  {'모델':<24} {'invoke 총':>10} {'stream 총':>10} {'첫 토큰까지':>12}")
for r in srows:
    ttft = f"{r['ttft']:.2f}초" if r["ttft"] is not None else "측정불가"
    print(f"  {r['name']:<24} {r['invoke_total']:>9.2f}초 "
          f"{r['stream_total']:>9.2f}초 {ttft:>12}")

print()
print("  총 시간 ≒ 변화 없음      → 스트리밍은 빨라지게 하지 않는다")
print("  첫 토큰까지 ≪ 총 시간    → 개선되는 것은 '체감 속도'")
print()
print("[정리] 스트리밍은 성능 최적화가 아니라 '사용자 경험' 개선입니다.")
print("       총 시간이 같아도 사용자는 훨씬 빠르다고 느낍니다.")

> **비동기 버전 (소개만)**
>
> ```python
> async for chunk in chain.astream(INPUTS):
>     print(chunk, end="", flush=True)
> ```
>
> 웹 서버처럼 여러 요청을 동시에 받아야 할 때 씁니다.
> 본 교과목에서는 직접 쓰지 않습니다. '이런 게 있다' 정도로 넘어갑니다.

## 오늘 확인할 것

- [ ] `ollama show` 로 파라미터 수·컨텍스트를 조회하고 **8GB 기준**으로 판정했다
- [ ] `ollama ps` 의 `SIZE`·`PROCESSOR` 를 계산값과 대조했다
- [ ] Modelfile 로 **나만의 챗봇**을 만들고, 코드 설정이 이기는 것을 **토큰 수**로 확인했다
- [ ] 같은 체인에서 **모델만** 갈아끼워 측정했다 ★★
- [ ] `with_fallbacks` 로 폴백이 걸리는 것을 확인했다
- [ ] `stream()` 으로 **첫 토큰까지**를 측정했다 ★
- [ ] **비교표 6항목**을 완성해 저장소에 커밋했다

> ⚠️ **Colab 값을 그대로 비교표에 적지 마십시오.**
> VRAM 판정은 **실습실 8GB 기준**, 속도는 **런타임이 GPU였는지 함께 기록**하십시오.